# Milestone 2 - Step 2: LLM-based Entity & Relationship Extraction

Unlike Milestone 2 - Step 1 (structured triples pulled directly from columns),
this notebook mines the **free-text `Ticket Description` field** using a local
LLM (Mistral, via Ollama) to extract `(Entity, Relation, Entity)` triples that
can't be captured by simple column mapping - e.g. the *specific* problem a
product is experiencing and the *specific* action needed to fix it.

## Step 0: Sanity-check the local LLM connection

In [ ]:
import ollama

response = ollama.chat(
    model="mistral",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print(response['message']['content'])

## Step 1: Prototype a triple-extraction prompt on a single example

In [ ]:
sample_text = """
Customer reported battery overheating issue in LG Smart TV.
Suggested resolution is factory reset.
"""

prompt = f"""
You are an AI system that extracts knowledge graph triples.

Extract triples strictly in this format:
(Entity, Relation, Entity)

Text:
{sample_text}

Only output triples. No explanations.
"""

response = ollama.chat(
    model="mistral",
    messages=[{"role": "user", "content": prompt}]
)

print(response['message']['content'])

## Step 2: Parse the LLM's raw text output into structured triples

In [ ]:
import re

raw_output = response['message']['content']

triples = re.findall(r"\((.*?)\)", raw_output)

parsed_triples = []

for triple in triples:
    parts = [part.strip() for part in triple.split(",")]
    if len(parts) == 3:
        parsed_triples.append(parts)

parsed_triples

## Step 3: Run extraction across the dataset

For each ticket, ask the LLM specifically for:
1. What problem the product is experiencing
2. What action is required to fix it

This keeps the LLM focused on two clean relation types instead of open-ended triples.

In [ ]:
import pandas as pd
import ollama
import re

# Load dataset
df = pd.read_excel("../data/processed/cleaned_tickets.xlsx")

llm_triples = []

for index, row in df.head(20).iterrows():

    text = row['Ticket Description']

    product = row['Product Purchased']

    prompt = f"""
The product involved is: {product}

From the text below, identify:

1. What specific problem the product is experiencing.
2. What action is required to fix it.

Return only valid triples in this format:
({product}, EXPERIENCING, Specific problem)
({product}, REQUIRED_ACTION, Specific action)

Do not create new entities.
Do not use 'User'.
Do not use placeholders.
If unclear, return nothing.

Text:
{text}
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}]
    )

    raw_output = response['message']['content']

    triples = re.findall(r"\((.*?)\)", raw_output)

    for triple in triples:
        parts = [part.strip() for part in triple.split(",")]

        if len(parts) == 3:
            subject, relation, obj = parts

            # Remove accidental labels
            subject = subject.replace("Product Name:", "").strip()
            relation = relation.replace("RELATION:", "").strip()
            obj = obj.replace("SPECIFIC DETAIL:", "").strip()

            # Filter weak outputs
            if obj.lower() in ["unknown", "unspecified", "issue"]:
                continue

            llm_triples.append([subject, relation, obj])

llm_triples

## Step 4: Convert to a DataFrame and save

In [ ]:
llm_triples_df = pd.DataFrame(llm_triples, columns=["Subject", "Predicate", "Object"])

llm_triples_df.head()

In [ ]:
llm_triples_df.to_csv("../data/processed/llm_triples.csv", index=False)

print("LLM triples saved successfully!")

## Step 5: Merge with structured triples

Combines the column-based triples from Milestone 2 - Step 1 with the
LLM-mined triples from this notebook into one unified triple set, which
becomes the single source of truth for graph construction.

In [ ]:
structured_df = pd.read_csv("../data/processed/structured_triples.csv")

final_triples_df = pd.concat([structured_df, llm_triples_df], ignore_index=True)

final_triples_df.to_csv("../data/processed/final_triples.csv", index=False)

print("Final merged triples saved successfully!")

## Outcome

`final_triples.csv` now contains both rule-based structured relationships and
LLM-mined free-text relationships in one file. This is what gets pushed into
Neo4j in the next step (Milestone 2 - Step 3, `scripts/push_to_neo4j.py`).